# Notebook 04 — Evaluation
Computes all metrics and generates cross-layer comparison plots.

Metrics:
- Reconstruction MSE and L0 sparsity (from training logs)
- Dead feature rate
- CLIP interpretability score (mean cosine similarity of top label)
- Cross-layer label distribution comparison

In [ ]:
import sys
sys.path.insert(0, '..')
import torch
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

ckpt4  = torch.load('../checkpoints/sae_layer4.pt', weights_only=False)
ckpt12 = torch.load('../checkpoints/sae_layer12.pt', weights_only=False)
labels4  = torch.load('../checkpoints/labels_layer4.pt', weights_only=False)
labels12 = torch.load('../checkpoints/labels_layer12.pt', weights_only=False)

log4  = ckpt4['log']
log12 = ckpt12['log']

In [ ]:
def last(log, key): return log[key][-1]

rows = [
    ('Final MSE',        last(log4,'mse'),       last(log12,'mse')),
    ('Final L0',         last(log4,'l0'),        last(log12,'l0')),
    ('Dead feature %',   last(log4,'dead_pct'),  last(log12,'dead_pct')),
    ('Mean CLIP score',  np.mean(labels4['scores']), np.mean(labels12['scores'])),
]
print(f"{'Metric':<25} {'Layer 4':>12} {'Layer 12':>12}")
print('-' * 50)
for name, v4, v12 in rows:
    print(f'{name:<25} {v4:>12.4f} {v12:>12.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, labels, title in [(axes[0], labels4, 'Layer 4'), (axes[1], labels12, 'Layer 12')]:
    top15 = Counter(labels['labels']).most_common(15)
    words, counts = zip(*top15)
    ax.barh(words, counts)
    ax.set_title(f'{title} — Top Labels')
    ax.set_xlabel('Feature count')
plt.tight_layout()
plt.savefig('../checkpoints/label_distribution.png', dpi=150)
plt.show()

In [ ]:
SCENE_WORDS   = {'sky','grass','water','ground','sand','snow','wall','floor',
                 'building','window','road','background'}
TEXTURE_WORDS = {'texture','pattern','edge','stripe','dot','grid','gradient'}

def categorize(label_list):
    scene   = sum(1 for l in label_list if l in SCENE_WORDS)
    texture = sum(1 for l in label_list if l in TEXTURE_WORDS)
    other   = len(label_list) - scene - texture
    return scene, texture, other

s4,  t4,  o4  = categorize(labels4['labels'])
s12, t12, o12 = categorize(labels12['labels'])

x = np.arange(3)
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - 0.2, [s4, t4, o4],   0.4, label='Layer 4')
ax.bar(x + 0.2, [s12, t12, o12], 0.4, label='Layer 12')
ax.set_xticks(x); ax.set_xticklabels(['Scene', 'Texture', 'Other'])
ax.set_ylabel('Feature count')
ax.set_title('Feature Semantics: Layer 4 vs Layer 12')
ax.legend()
plt.tight_layout()
plt.savefig('../checkpoints/cross_layer_semantics.png', dpi=150)
plt.show()

print(f'Layer 4  — Scene: {s4}, Texture: {t4}, Other: {o4}')
print(f'Layer 12 — Scene: {s12}, Texture: {t12}, Other: {o12}')

## Human Evaluation Protocol

Sample 30 features per layer (60 total).
For each feature:
  1. Show the top-10 maximally activating image patches
  2. Show the CLIP-assigned label
  3. Rate label accuracy 1–5:
       5 = label perfectly describes what patches share
       3 = loosely related
       1 = unrelated

Report mean ± std per layer. This provides the qualitative validation
required by the P3 spec alongside the automated CLIP scores.